# Coming from Spinach

spinpy keeps Spinach's structure: the same functions, called in the same
order, with the same arguments, field names, units, and tolerances. A
Spinach script translates into spinpy almost line by line. This page lists,
topic by topic, what carries over unchanged and what is different, and
what is not ported yet. The [user guide](user-guide.ipynb) explains each
function in full.

Every Python block on this page is executed by `tests/docs/`. MATLAB blocks
are Spinach code, shown for comparison and not run. The companion notebook
is [`notebooks/coming-from-spinach.ipynb`](coming-from-spinach.ipynb).

*Generated from [`docs/coming-from-spinach.md`](../coming-from-spinach.md) by `docs/build_notebooks.py`; edit the Markdown guide, not this notebook.*

Run the cells in order: later cells use names defined by earlier ones. Plotting cells need matplotlib (`pip install -e ".[docs]"`).

## Contents

1. [A Spinach script, translated](#A-Spinach-script,-translated)
2. [General differences](#General-differences)
3. [Spin system: `sys` and `inter`](#Spin-system:-sys-and-inter)
4. [Basis: `bas`](#Basis:-bas)
5. [Operators and states](#Operators-and-states)
6. [Assumptions and the Hamiltonian](#Assumptions-and-the-Hamiltonian)
7. [Experiment parameters](#Experiment-parameters)
8. [Calculation modalities](#Calculation-modalities)
9. [Processing and plotting](#Processing-and-plotting)
10. [Messages and errors](#Messages-and-errors)

## A Spinach script, translated

Spinach's `examples/nmr_liquids/pa_styrene.m` simulates the 1H spectrum of
styrene at 7.046 T:

```matlab
sys.isotopes={'1H','1H','1H','1H','1H','1H','1H','1H'};
sys.magnet=7.046;
inter.zeeman.scalar={6.720  6.720  6.587  6.587  6.527  6.063   5.085  4.555};
inter.coupling.scalar={ 0.0  1.9170  7.7884  0.6061  1.2447 -0.5347  0.0390  0.1582
                        0.0  0.0     0.6061  7.7884  1.2447 -0.5347  0.0390  0.1582
                        0.0  0.0     0.0     1.4184  7.4390  0.3693 -0.0416  0.0169
                        0.0  0.0     0.0     0.0     7.4390  0.3693 -0.0416  0.0169
                        0.0  0.0     0.0     0.0     0.0    -0.2279  0.2349  0.2905
                        0.0  0.0     0.0     0.0     0.0     0.0    17.6002 10.8954
                        0.0  0.0     0.0     0.0     0.0     0.0     0.0     1.0382
                        0.0  0.0     0.0     0.0     0.0     0.0     0.0     0.0};
bas.formalism='zeeman-hilb';
bas.approximation='none';
spin_system=create(sys,inter);
spin_system=basis(spin_system,bas);
parameters.spins={'1H'};
parameters.rho0=state(spin_system,'L+','1H');
parameters.coil=state(spin_system,'L+','1H');
parameters.decouple={};
parameters.offset=1700;
parameters.sweep=1000;
parameters.npoints=16384;
parameters.zerofill=65536;
parameters.axis_units='ppm';
parameters.invert_axis=1;
fid=liquid(spin_system,@acquire,parameters,'nmr');
fid=apodisation(spin_system,fid,{{'gauss',6}});
spectrum=fftshift(fft(fid,parameters.zerofill));
kfigure(); plot_1d(spin_system,real(spectrum),parameters);
```

In spinpy:

In [ ]:
import numpy as np
import spinpy

sys = {"isotopes": ["1H"] * 8, "magnet": 7.046}
inter = {
    "zeeman": {"scalar": [6.720, 6.720, 6.587, 6.587, 6.527, 6.063, 5.085, 4.555]},
    "coupling": {"scalar": [
        [0.0, 1.9170, 7.7884, 0.6061, 1.2447, -0.5347,  0.0390,  0.1582],
        [0.0, 0.0,    0.6061, 7.7884, 1.2447, -0.5347,  0.0390,  0.1582],
        [0.0, 0.0,    0.0,    1.4184, 7.4390,  0.3693, -0.0416,  0.0169],
        [0.0, 0.0,    0.0,    0.0,    7.4390,  0.3693, -0.0416,  0.0169],
        [0.0, 0.0,    0.0,    0.0,    0.0,    -0.2279,  0.2349,  0.2905],
        [0.0, 0.0,    0.0,    0.0,    0.0,     0.0,    17.6002, 10.8954],
        [0.0, 0.0,    0.0,    0.0,    0.0,     0.0,     0.0,     1.0382],
        [0.0, 0.0,    0.0,    0.0,    0.0,     0.0,     0.0,     0.0],
    ]},
}
bas = {"formalism": "zeeman-hilb", "approximation": "none"}
spin_system = spinpy.create(sys, inter)
spin_system = spinpy.basis(spin_system, bas)

parameters = {
    "spins": ["1H"],
    "rho0": spinpy.state(spin_system, "L+", "1H"),
    "coil": spinpy.state(spin_system, "L+", "1H"),
    "decouple": [],
    "offset": 1700,
    "sweep": 1000,
    "npoints": 2048,     # Spinach example: 16384, see below
    "zerofill": 8192,    # Spinach example: 65536
}
fid = spinpy.liquid(spin_system, spinpy.acquire, parameters, "nmr")

# apodisation(spin_system,fid,{{'gauss',6}}): halve the first point, Gaussian window
fid[0] = fid[0] / 2
fid = fid * np.exp(-6 * np.linspace(0, 1, fid.size) ** 2)
spectrum = np.fft.fftshift(np.fft.fft(fid, parameters["zerofill"]))

assert spectrum.shape == (8192,)

The run uses 2048 points instead of 16384 to keep this page quick: with
eight spins the Hilbert space has dimension 256, and the full-length run
takes about 30 s on four CPU cores. Set `"npoints": 16384` and
`"zerofill": 65536` to reproduce the Spinach example exactly.

`plot_1d` is not ported; the ppm axis and the plot take a few lines (see
[Processing and plotting](#Processing-and-plotting)):

In [ ]:
import matplotlib.pyplot as plt
from spinpy.isotopes import spin

def ppm_axis(spin_system, parameters, isotope="1H"):
    """Spinach's axis_1d with axis_units='ppm', for one channel."""
    npts = parameters["zerofill"]
    hz = parameters["offset"] + np.fft.fftshift(np.fft.fftfreq(npts, 1 / parameters["sweep"]))
    carrier = -spin(isotope)[0] * spin_system.magnet / (2 * np.pi)
    return -1e6 * hz / carrier

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ppm_axis(spin_system, parameters), spectrum.real)
ax.invert_xaxis()                      # parameters.invert_axis=1
ax.set_xlabel("1H chemical shift / ppm")
ax.set_yticks([])
plt.show()

## General differences

| topic | Spinach | spinpy |
|---|---|---|
| spin numbers | 1-based | **0-based**: Spinach spin 1 is spinpy spin 0 |
| input structures | MATLAB structs: `sys.magnet=14.1;` | dictionaries: `sys = {"magnet": 14.1}` |
| lists | cell arrays `{'1H','13C'}` | lists `["1H", "13C"]` |
| spin system object | struct, copied and returned by each function | frozen dataclass, copied and returned by each function |
| matrix product | `A*B` | `A @ B` (`A * B` is element-wise for NumPy and SciPy arrays) |
| conjugate transpose | `A'` | `A.conj().T` |
| sparse matrices | MATLAB sparse | `scipy.sparse.csr_array` |
| time-domain results | `n x 1` column vectors | 1-D arrays of length `n` |
| function handles | `@acquire` | `spinpy.acquire` |
| units | ppm, Hz, Tesla, Bohr magneton; rad/s internally | the same |
| tolerances | `sys.tols.<name>` | `sys["tols"]["<name>"]`, same names and defaults |

The `*` versus `@` difference is the most common translation mistake,
because both run without error:

In [ ]:
Lp = spinpy.operator(spin_system, "L+", "1H")
Lm = spinpy.operator(spin_system, "L-", "1H")
matrix_product = Lp @ Lm                 # MATLAB Lp*Lm
elementwise = Lp * Lm                    # MATLAB Lp.*Lm, a different matrix
assert (matrix_product != elementwise).nnz > 0

## Spin system: `sys` and `inter`

`create(sys, inter)` reads the same fields as in Spinach, with the same
meaning and units. Shown for a 1H-13C pair:

```matlab
sys.isotopes={'1H','13C'};
sys.magnet=14.1;
inter.zeeman.scalar={2.0 30.0};
inter.coupling.scalar=cell(2,2);
inter.coupling.scalar{1,2}=145;
spin_system=create(sys,inter);
```

In [ ]:
sys = {"isotopes": ["1H", "13C"], "magnet": 14.1}
inter = {
    "zeeman": {"scalar": [2.0, 30.0]},
    "coupling": {"scalar": [[None, 145.0],
                            [None, None]]},
}
hc = spinpy.create(sys, inter)

- **Same:** isotope names (from Spinach's `spin.m`, including `'E'` for
  electrons), units (ppm for nuclear shifts, Bohr magneton units for
  g-values, Hz for couplings, Tesla for the field), the rule that a
  coupling entered both above and below the diagonal is summed, dropping
  of interactions below `tols.inter_cutoff`, and the internal storage of
  every interaction as a 3x3 tensor in rad/s.
- **Empty entries:** where Spinach uses empty cells (`cell(2,2)`), use
  `None` or `0`. A nested list, or a NumPy array for scalar couplings,
  are both accepted.
- **Tolerances:** `sys["tols"] = {"prop_chop": 1e-14}` does what
  `sys.tols.prop_chop=1e-14` does. The values are in `spin_system.tols`.
- **Immutability:** Spinach scripts that change a spin system after
  `create` (`spin_system.inter.coupling.matrix{1,2}=...`) must build a new
  system instead. A parameter scan is a loop over `create`:

In [ ]:
for J in (140.0, 145.0, 150.0):
    scanned = spinpy.create(sys, {**inter, "coupling": {"scalar": [[0, J], [0, 0]]}})
    assert np.isclose(scanned.coupling_matrix[0][1][0, 0], 2 * np.pi * J)

**Not ported yet** (each raises `NotImplementedError` naming the field):
`sys.output`, `sys.enable` (GPU and other options), `sys.disable`,
`sys.parallel`, `sys.scratch`, `inter.coordinates` and `inter.pbc`
(dipolar couplings), `inter.zeeman.eigs`/`euler`,
`inter.coupling.eigs`/`euler`, `inter.giant`, `inter.suscept`,
`inter.order_matrix`, `inter.relaxation` and all relaxation fields,
`inter.chem` (kinetics), `inter.temperature`, and bosonic modes
(cavities, phonons, transmons).

## Basis: `bas`

In [ ]:
bas = {"formalism": "zeeman-hilb", "approximation": "none"}
hc = spinpy.basis(hc, bas)

- **Same:** field names; `zeeman-hilb` requires `approximation='none'`;
  basis states are Zeeman product states ordered as `kron(spin 1, spin 2, ...)`,
  so matrices are element for element identical to Spinach's.
- **Different:** none for `zeeman-hilb`.
- **Not ported yet:** the `zeeman-liouv` and `sphten-liouv` formalisms, and
  with them every truncation field (`level`, `space_level`,
  `connectivity`, `longitudinals`, `projections`, `zero_quantum`, `manual`,
  the IK approximations) and permutation symmetry (`sym_group`,
  `sym_spins`). In Spinach, switching a script from Hilbert to Liouville
  space changes only `bas`, and states become vectors instead of matrices;
  spinpy will keep that behaviour when the Liouville formalisms are ported.

## Operators and states

The three Spinach call forms carry over; only the brackets and the spin
numbers change:

| Spinach | spinpy | result |
|---|---|---|
| `operator(spin_system,'Lz','1H')` | `spinpy.operator(ss, "Lz", "1H")` | sum of Lz over all 1H |
| `operator(spin_system,'Lz',[1 3])` | `spinpy.operator(ss, "Lz", [0, 2])` | Lz(1) + Lz(3) in Spinach numbering |
| `operator(spin_system,{'Lz','L+'},{1,2})` | `spinpy.operator(ss, ["Lz", "L+"], [0, 1])` | product Lz(1) L+(2) |
| `operator(spin_system,'Lx','all')` | `spinpy.operator(ss, "Lx", "all")` | also `'nuclei'`, `'electrons'` |
| `state(spin_system,'L+','13C')` | `spinpy.state(ss, "L+", "13C")` | density matrix |

In Spinach a cell array of spin numbers, `{1,2}`, selects the product form.
In spinpy the type of the *first* argument does: a string gives a sum, a
list gives a product.

In [ ]:
Lz_sum = spinpy.operator(hc, "Lz", [0, 1])         # Lz(H) + Lz(C)
LzSz = spinpy.operator(hc, ["Lz", "Lz"], [0, 1])   # Lz(H) Lz(C)
assert np.allclose((LzSz @ LzSz).toarray(), np.eye(4) / 16)

- **Same:** operator labels `E`, `Lx`, `Ly`, `Lz`, `L+`, `L-`, `Tl,m`;
  matrix elements (Spinach's `Lz` for spin-1/2 is `0.4999999999999999`,
  one ulp below 0.5, and spinpy reproduces it); in Hilbert space a state
  is the operator itself, unnormalised.
- **Different:** `operator` has no `operator_type` or `format` arguments,
  and `state` has no `method` argument. Spinach ignores all three in
  Hilbert space.
- **Not ported yet:** labels `CT*` (central transition) and `ZL*`
  (Zeeman energy level projectors), and operators on bosonic modes.

## Assumptions and the Hamiltonian

In [ ]:
H = spinpy.hamiltonian(spinpy.assume(hc, "nmr"))
H_zeeman = spinpy.hamiltonian(spinpy.assume(hc, "nmr", "zeeman"))
assert np.allclose(H - H_zeeman, 2 * np.pi * 145.0 * LzSz.toarray())  # weak 1H-13C coupling

- **Same:** `assume(spin_system, 'nmr')` and the retention options
  `'zeeman'` and `'couplings'`; the `'nmr'` truncation rules (strong
  coupling within an isotope, weak `LzSz` coupling between isotopes);
  rounding of every element to `tols.liouv_zero`; the dense-or-sparse
  choice by `tols.small_matrix` and `tols.dense_matrix`.
- **Different:** Spinach's `[H,Q]=hamiltonian(...)` also returns the
  anisotropic components `Q`; spinpy returns `H` alone. Spinach sums the
  Hamiltonian terms in a random order, so its last digits change from run
  to run; spinpy sums in a fixed order and is reproducible.
- **Not ported yet:** the other assumption sets (`'esr'`, `'deer'`,
  `'labframe'`, `'qnmr'`, the DNP sets, ...), `operator_type` (`'left'`,
  `'right'`, `'comm'`), and anisotropic interactions.

## Experiment parameters

`parameters` is a dictionary with Spinach's field names. `liquid` copies it
before filling in defaults, so the caller's dictionary is never changed,
as with MATLAB's value semantics.

| field | meaning | ported |
|---|---|---|
| `spins` | isotopes of the working channels, e.g. `["1H"]` | yes |
| `offset` | transmitter offset per channel, Hz; a number or a list | yes; default 0 |
| `sweep` | spectral width, Hz | yes |
| `npoints` | number of complex points | yes |
| `rho0`, `coil` | initial state, detection state | yes |
| `decouple` | isotopes to decouple | only empty |
| `rframes`, `needs` | extra rotating frames, context requests | only empty |
| `homodec_oper`, `homodec_pwr`, `dead_time` | `acquire` options | no |
| `zerofill`, `axis_units`, `invert_axis` | read by Spinach's plotting functions | stored, unused |

Fields that only Spinach's processing and plotting functions read are
accepted and ignored by the simulation, as in Spinach.

## Calculation modalities

| Spinach | spinpy |
|---|---|
| contexts: `liquid`, `powder`, `crystal`, `singlerot`, `doublerot`, `floquet`, `gridfree`, `imaging`, ... | `liquid` only |
| pulse sequences in `experiments/`: `acquire`, `hsqc`, `cosy`, `noesy`, ... | `acquire`; write others yourself |
| `evolution` outputs `'observable'`, `'final'`, `'total'`, `'trajectory'`, `'refocus'`, `'multichannel'` | `'observable'` |
| `evolution(...,destination)` destination state screening | not ported |
| `step`, `krylov`, `pulse`, `shaped_pulse_*` | not ported; use `propagator` for ideal pulses |
| `propagator`: Pade `expm` below `small_matrix`, scaled and squared Taylor series above | the same |
| relaxation (`relaxation`), chemical kinetics (`kinetics`) | not ported; `R` and `K` are zero |
| parallel `evolution` (Kuprov-Edwards split), `parfor` | serial; the same numbers |
| GPU via `sys.enable={'gpu'}` | CPU only; CuPy backend planned |

**Pulse sequences** have Spinach's signature, `sequence(spin_system,
parameters, H, R, K)`, and are passed to `liquid` the same way. Inside,
use `spinpy.operator`, `spinpy.propagator`, and `spinpy.evolution` as you
would in MATLAB; the user guide has a
[worked example](user-guide.ipynb#Writing-your-own-pulse-sequence).
Spinach sequences usually combine the generator as `L = H + 1i*R + 1i*K`;
in Python that is `L = H + 1j * R + 1j * K`.

**Numerical agreement** with Spinach is checked stage by stage against
stored Spinach 2.13 output; see
[Checking results against Spinach](user-guide.ipynb#Checking-results-against-Spinach).

## Processing and plotting

Spinach's processing and plotting helpers are not ported; NumPy and
matplotlib do the same in a line or two.

| Spinach | NumPy / matplotlib |
|---|---|
| `fftshift(fft(fid,zerofill))` | `np.fft.fftshift(np.fft.fft(fid, zerofill))` |
| `apodisation(spin_system,fid,{{'exp',k}})` | `fid[0] /= 2; fid *= np.exp(-k * np.linspace(0, 1, fid.size))` |
| `apodisation(spin_system,fid,{{'gauss',k}})` | `fid[0] /= 2; fid *= np.exp(-k * np.linspace(0, 1, fid.size) ** 2)` |
| `axis_1d`, `ft_axis` | `offset + np.fft.fftshift(np.fft.fftfreq(zerofill, 1 / sweep))` |
| `plot_1d(...)` with `axis_units='ppm'`, `invert_axis=1` | `ax.plot(ppm, spectrum.real); ax.invert_xaxis()` |
| `kfigure()` | `plt.subplots()` |

Spinach's `apodisation` halves the first point of the FID by default;
apply that step yourself when comparing spectra with Spinach's. The
`ppm_axis` function in the [translated script](#A-Spinach-script,-translated)
converts Hz to ppm the way `axis_1d` does.

## Messages and errors

- Spinach prints a running log (`report`) and summary tables. spinpy is
  silent; warnings, such as a time step much longer than the system's
  dynamics, go to Python's `logging` module under the name `"spinpy"`.
- Spinach's input checks (`grumble`) become `ValueError` for invalid input.
- Anything that Spinach supports and spinpy does not yet raises
  `NotImplementedError` with a message naming the missing piece. spinpy
  never silently ignores an input it does not understand.

In [ ]:
try:
    spinpy.create({"isotopes": ["1H"], "magnet": 14.1, "output": "hush"}, {})
except NotImplementedError as error:
    print(error)